<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day5/Session1/session1_first_api_call.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 1 — What Is an API, and Your First Call

**Day 5 · 09:00 – 10:30**

Yesterday you ran a model on your own GPU. Today you borrow a much larger one — free,
in five minutes, with no GPU at all.

## Before you start

1. Go to **<https://aistudio.google.com/apikey>** and sign in with any Google account
2. Click **Create API key** (no credit card required)
3. **Copy the key** — it starts with `AIzaSy...` and is shown clearly only once

Do **not** paste it into a code cell. The next cell shows the safe way.

## Cell 1 — Store your key safely

> ### Never put an API key in your code.
> People run automated scanners over public GitHub repositories and find leaked keys
> within **minutes**. A leaked key gets used by strangers — and if billing is ever
> enabled, you pay for it.

**Do this instead:**

1. Click the **key icon** in the Colab left sidebar (*Secrets*)
2. Click **Add new secret**
3. Name it exactly: `GEMINI_API_KEY`
4. Paste your key as the value
5. Turn **on** *Notebook access*

In [ ]:
!pip install -q -U google-genai

import os
from google import genai
from google.genai import types

# Load the key from Colab Secrets (key icon in the left sidebar)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    # Not on Colab, or secret not set - fall back to typing it in
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client()          # reads GEMINI_API_KEY from the environment
MODEL = "gemini-3.5-flash"       # see the model-listing cell if this name fails

print("Client ready. Key length:", len(os.environ["GEMINI_API_KEY"]))

If that printed a key length of about 39, you are ready.

If you are not on Colab, the cell falls back to a hidden password prompt — the key still
never appears in the notebook.

## Cell 2 — Which models can your key actually use?

Model names change often. Rather than trusting any tutorial (including this one), ask
the API directly.

In [ ]:
available = []
for m in client.models.list():
    actions = getattr(m, "supported_actions", None) or []
    if not actions or "generateContent" in actions:
        available.append(m.name)

print(f"{len(available)} models available to your key\n")
for name in available[:25]:
    print("  ", name)

Look for something containing **`flash`** — those are the fast, free-tier-friendly
models. If `gemini-3.5-flash` is not in the list, change the `MODEL` variable in Cell 1
to a name that IS listed, and re-run.

## Cell 3 — Your first API call

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain what an API is, in two sentences, for a complete beginner.",
)

print(response.text)

**That is the whole thing.**

| Part | What it does |
|---|---|
| `client` | who you are (your key) |
| `model` | which brain to use |
| `contents` | what you are asking |
| `response.text` | the answer that came back |

No GPU. No 13 GB download. No OOM crash. You just used a model far larger than anything
that fits on a free T4.

## Cell 4 — Ask it something in Amharic

Compare this with the fine-tuned 0.5B model from yesterday.

In [ ]:
questions = [
    "የኢትዮጵያ ዋና ከተማ ማን ናት?",
    "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?",
]

for q in questions:
    r = client.models.generate_content(model=MODEL, contents=q)
    print("Q:", q)
    print("A:", r.text[:300])
    print("-" * 60)

### Think about what you are seeing

This model handles Amharic far better than the 0.5B model you fine-tuned yesterday.
It is also **hundreds of times larger**, runs on Google's hardware, and needs an
internet connection.

Neither approach is "better". They answer different questions:

| | API (today) | Self-hosted (Day 4) |
|---|---|---|
| Quality | Much higher | Limited by model size |
| Works offline | ❌ No | ✅ Yes |
| Data leaves your building | ⚠️ Yes | ✅ No |
| Cost | Grows per request | Fixed once you own the hardware |
| Setup time | 5 minutes | A day |

## Cell 5 — How many tokens did that cost?

APIs charge and rate-limit by **tokens**, not words — exactly the tokens you studied
on Day 4.

In [ ]:
prompt = "Write one paragraph about the highlands of Ethiopia."

r = client.models.generate_content(model=MODEL, contents=prompt)

usage = r.usage_metadata
print("Input tokens :", usage.prompt_token_count)
print("Output tokens:", usage.candidates_token_count)
print("Total        :", usage.total_token_count)
print()
print(r.text)

### Try this

Change the prompt to Amharic and re-run. Watch the **input token count** rise for
roughly the same meaning — the 3x penalty you measured yesterday, now costing you real
quota.

## Cell 6 — A reusable helper with error handling

Free tiers have rate limits. A production-quality helper retries politely instead of
crashing.

In [ ]:
import time

def ask(prompt, model=MODEL, system=None, tries=4):
    """Send a prompt. Retry with backoff on rate limits. Return the text."""
    config = None
    if system:
        config = types.GenerateContentConfig(system_instruction=system)

    for attempt in range(tries):
        try:
            r = client.models.generate_content(
                model=model, contents=prompt, config=config)
            return r.text
        except Exception as e:
            msg = str(e)
            is_rate_limit = "429" in msg or "RESOURCE_EXHAUSTED" in msg
            if is_rate_limit and attempt < tries - 1:
                wait = 2 ** attempt          # 1s, 2s, 4s
                print(f"  rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                return f"ERROR: {msg[:200]}"
    return "ERROR: gave up after retries"


print(ask("Name three regions of Ethiopia. Just the names."))

## Cell 7 — Your turn

Ask it three things that would be useful in your own work.

In [ ]:
my_questions = [
    "Write your own question here.",
    "And another.",
    "And a third.",
]

for q in my_questions:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 60)

---

## Checklist

- [ ] I created an API key at aistudio.google.com/apikey
- [ ] I stored it in **Colab Secrets**, not in a code cell
- [ ] I listed the models my key can use
- [ ] I sent a prompt and got a real answer
- [ ] I saw the token counts for a request
- [ ] I have a helper function that survives a rate-limit error

## If something goes wrong

| Error | Fix |
|---|---|
| `SecretNotFoundError` | The secret name must be exactly `GEMINI_API_KEY`, with notebook access ON |
| `API key not valid` | Copy the key again — no spaces, no line breaks |
| `404` / model not found | Run Cell 2 and use a model name from that list |
| `429 RESOURCE_EXHAUSTED` | Rate limit. Wait a minute, or use the `ask()` helper |
| `ModuleNotFoundError: google.genai` | Run the install cell first |

**Next:** Session 2 — controlling the model properly.